In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import warnings

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [12]:
# Load
df = pd.read_excel('/content/Exercise 15 Regression Trees Strength2.xlsx')
X = df.drop(columns=['Strength2'])
y = df['Strength2']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature scaling for ANN
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled  = x_scaler.transform(X_test)

In [13]:
# Models
alphas = np.logspace(-3, 3, 25)
models = {
    "Linear Regression": LinearRegression(),
    "Ridge (CV)": RidgeCV(alphas=alphas),
    "Lasso (CV)": LassoCV(alphas=alphas, max_iter=10000, cv=5),
    "ElasticNet (CV)": ElasticNetCV(alphas=alphas, l1_ratio=[0.1, 0.5, 0.9], max_iter=10000, cv=5),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost Regressor": XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=6, subsample=1.0,
        colsample_bytree=1.0, reg_lambda=1.0, reg_alpha=0.0, random_state=42
    ),
}

results = []
test_preds = {}

for name, model in models.items():
    # Use scaled X for all to align with ANN
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results.append([name, round(r2, 4), round(rmse, 4)])
    test_preds[name] = np.asarray(y_pred).ravel()

In [14]:
# Scale target
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

# ANN
model_ann = Sequential([
    Dense(512, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='linear')
])
model_ann.compile(optimizer='adam', loss='mse')

early_stop = EarlyStopping(monitor='loss', patience=15, restore_best_weights=True)

model_ann.fit(X_train_scaled, y_train_scaled, epochs=300, batch_size=64, verbose=1, callbacks=[early_stop])

# Predictions (invert target scaling)
y_pred_scaled = model_ann.predict(X_test_scaled)
y_pred_ann = scaler_y.inverse_transform(y_pred_scaled).ravel()

r2_ann = r2_score(y_test, y_pred_ann)
rmse_ann = np.sqrt(mean_squared_error(y_test, y_pred_ann))

results.append(["ANN", round(r2_ann, 4), round(rmse_ann, 4)])
test_preds["ANN"] = y_pred_ann

print(f"\nANN R²: {r2_ann:.4f}")
print(f"ANN RMSE: {rmse_ann:.4f}")

Epoch 1/300


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.3824
Epoch 2/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2109
Epoch 3/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1750
Epoch 4/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1444
Epoch 5/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1168
Epoch 6/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1088
Epoch 7/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0905
Epoch 8/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.1162
Epoch 9/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.1003
Epoch 10/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0898
Epoch 11/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0635
Epoch 12/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0586
Epoch 13/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0577
Epoch 14/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0788
Epoch 15/300
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0792
Epoch 16/300
5

In [15]:
# Display results
results_df = pd.DataFrame(results, columns=["Model", "Test R² Score", "Test RMSE"])
print("\nModel Performance Comparison:\n")
print(results_df)


Model Performance Comparison:

               Model  Test R² Score  Test RMSE
0  Linear Regression         0.7243   186.6320
1         Ridge (CV)         0.8147   153.0146
2         Lasso (CV)         0.8145   153.0956
3    ElasticNet (CV)         0.8136   153.4834
4      Decision Tree         0.6913   197.5016
5      Random Forest         0.8293   146.8659
6  XGBoost Regressor         0.8507   137.3518
7                ANN         0.8519   136.7964
